This notebook allows you to generate pan-India rasters for the following:
1. LTP (Large Tree Patch, contiguous tree cover > 1 ha) mask
2. Area and perimeter of LTPs
3. Distance from forest (LTP) edge
4. Distance of a given LTP to its nearest neighbor
5. Historical tree cover masks (further details in the Forest Cover Masks section)

Run Set up, Config and Utils every time - then run whichever section you need. Scale and year are set in Config.

# Set up

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import ee
import geemap
import pandas as pd
import re
import time

ee.Authenticate()
ee.Initialize(project='ee-ictd-dhruvi')

# Config

In [3]:
err = ee.ErrorMargin(100)
scale = 100
year = 2023

# india = ee.FeatureCollection('FAO/GAUL/2015/level0').filter(ee.Filter.eq('ADM0_NAME', 'India'))

kolar = ee.FeatureCollection('FAO/GAUL_SIMPLIFIED_500m/2015/level2').filter(ee.Filter.eq("ADM2_NAME","Kolar"))
karnataka = ee.FeatureCollection('FAO/GAUL/2015/level1').filter(ee.Filter.eq('ADM1_NAME', "Karnataka")).first()

roi = kolar
indiaDistricts = ee.FeatureCollection("projects/ee-indiasat/assets/india_district_boundaries")
indiaACZs = ee.FeatureCollection("projects/ee-mtpictd/assets/harsh/Agroclimatic_regions")
india = indiaACZs

In [4]:
agroClimaticZones = [
    'Eastern Plateau & Hills Region',
    'Southern Plateau and Hills Region',
    'East Coast Plains & Hills Region',
    'Western Plateau and Hills Region',
    'Central Plateau & Hills Region',
    'Lower Gangetic Plain Region',
    'Middle Gangetic Plain Region',
    'Eastern Himalayan Region',
    'Western Himalayan Region',
    'Upper Gangetic Plain Region',
    'Trans Gangetic Plain Region',
    'West Coast Plains & Ghat Region',
    'Gujarat Plains & Hills Region',
    'Western Dry Region'
]
aczAcronymDict = {'Eastern Plateau & Hills Region': 'EPAHR',
                  'Southern Plateau and Hills Region': 'SPAHR',
                  'East Coast Plains & Hills Region': 'ECPHR',
                  'Western Plateau and Hills Region': 'WPAHR',
                  'Central Plateau & Hills Region': 'CPAHR',
                  'Lower Gangetic Plain Region': 'LGPR',
                  'Middle Gangetic Plain Region': 'MGPR',
                  'Eastern Himalayan Region': 'EHR',
                  'Western Himalayan Region': 'WHR',
                  'Upper Gangetic Plain Region': 'UGPR',
                  'Trans Gangetic Plain Region': 'TGPR',
                  'West Coast Plains & Ghat Region': 'WCPGR',
                  'Gujarat Plains & Hills Region': 'GPHR',
                  'Western Dry Region': 'WDR'
                  }

# Utils

In [5]:
# Dynamic World
def get_dw_tree_cover(aoi, start_date, end_date, scale = 10):
  tree_cover_dw = ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1").filterDate(start_date, end_date) \
                .filterBounds(aoi).select('label').mode().clip(aoi)
  return tree_cover_dw.updateMask(tree_cover_dw.eq(1)).reproject(crs='EPSG:4326', scale=scale).rename('tree_cover')

# IndiaSAT (defined for given agricultural year)
def get_is_tree_cover(aoi, year, scale = 10):
  year = int(year)
  indiasat_asset = f"projects/corestack-datasets/assets/datasets/LULC_v3_river_basin/pan_india_lulc_v3_{year}_{year+1}"
  lulc_image = ee.Image(indiasat_asset).select("predicted_label").clip(aoi)
  return lulc_image.updateMask(lulc_image.eq(6)).reproject(crs='EPSG:4326', scale=scale).rename('tree_cover')

# Union of Dynamic World and IndiaSAT
def get_tree_cover(aoi, year, scale = 10):
  # year = 2023 would take data from July 2023 - June 2024 (AY 2023)
  year = int(year)
  start_date = ee.Date(f'{year}-07-01')
  end_date = ee.Date(f'{year+1}-06-30')

  tree_cover_is = get_is_tree_cover(aoi, year, scale)
  tree_cover_dw = get_dw_tree_cover(aoi, start_date, end_date, scale)
  tree_cover = tree_cover_is.mask().Or(tree_cover_dw.mask())
  tree_cover = tree_cover.updateMask(tree_cover)
  return tree_cover.reproject(crs='EPSG:4326', scale=scale)

In [6]:
# Returns Image Collection containing all images in folder_path
def get_img_collection(folder_path):
  asset_list = ee.data.listAssets({'parent': folder_path})['assets']
  image_list = [ee.Image(asset['name']) for asset in asset_list]
  return ee.ImageCollection(image_list)

# Returns state boundary from GAUL's collection
# Not every state is defined in GAUL 2015, e.g. Telangana
def get_state_roi(state):
  gaulName = gaulNames[state]
  roi = ee.FeatureCollection('FAO/GAUL/2015/level1') \
          .filter(ee.Filter.eq('ADM1_NAME', gaulName)).first() \
          .set('state', state)
  return roi

# Export LTP mask, area and perimeter rasters district-wise

Functions for computing LTP mask, area, perimeter

In [ ]:
# Computes the feature's geometry area and perimeter, adds it as a property
def add_area_perimeter(feature):
    patch_area = ee.Number(feature.geometry(err).area(err)) # keep area in m2
    patch_perimeter = ee.Number(feature.geometry(err).perimeter(err))   # meters

    ltp = ee.Algorithms.If(patch_area.gte(ee.Number(1e4)), 1, 0)
    return feature.set({"area_m2": patch_area, "perimeter_m": patch_perimeter, "large_tree_patch": ltp})

def getLTPImage(roi, year, scale):
  tree_cover = get_tree_cover(roi.geometry(), year, scale)
  tree_patches = tree_cover.reduceToVectors(crs='EPSG:4326', scale=scale, geometry=roi.geometry(), maxPixels=1e13, geometryType='polygon', bestEffort=True)
  tree_patches = tree_patches.map(add_area_perimeter)
  ltp_mask_img = tree_patches.reduceToImage(
    properties=['large_tree_patch'],
    reducer=ee.Reducer.first()
  ).rename('large_tree_patch')

  area_img = tree_patches.reduceToImage(
    properties=['area_m2'],
    reducer=ee.Reducer.first()
  ).rename('area_m2')

  perimeter_img = tree_patches.reduceToImage(
    properties=['perimeter_m'],
    reducer=ee.Reducer.first()
  ).rename('perimeter_m')

  ltp_img = ltp_mask_img
  ltp_img = ltp_img.addBands([area_img, perimeter_img])
  ltp_img = ltp_img.updateMask(ltp_mask_img).reproject(crs='EPSG:4326', scale=scale).clip(roi.geometry())
  return ltp_img



Loop to export LTP mask, area, perimeter raster district by district

In [ ]:
distRoiList = []
for acz in agroClimaticZones:
    print(acz)
    distDf = pd.read_csv(f'drive/MyDrive/Anoushka/ACZ_Districts/{acz}.csv')
    distList = list(distDf['Name'])
    for i, district in enumerate(distList):
      # print(i, district)
      geom = ee.Feature(indiaDistricts.filter(ee.Filter.eq('Name', district)).first())

      # Clean up district name
      district = district.replace('&', 'and').replace('.', '')
      district = re.sub(r'[^a-zA-Z0-9]', '_', district)

      ltp_img = getLTPImage(geom, year, scale)
      task = ee.batch.Export.image.toAsset(
          image=ltp_img,
          description=f'LTP_{district}',
          assetId=f'projects/ee-mtpictd/assets/anoushka/LTP_area_perim_2023/{district}',
      )
      task.start()
      print(f'Export task started for {district}.')
      time.sleep(0.25)

Eastern Plateau & Hills Region
Export task started for East_Godavari.
Export task started for Srikakulam.
Export task started for Visakhapatnam.
Export task started for Vizianagaram.
Export task started for AurangabadB.
Export task started for Banka.
Export task started for Gaya.
Export task started for Jamui.
Export task started for Nawada.
Export task started for Rohtas.
Export task started for Baloda_Bazar.
Export task started for Balod.
Export task started for BalrampurC.
Export task started for Bastar.
Export task started for Bemetara.
Export task started for BijapurC.
Export task started for BilaspurC.
Export task started for Dantewada.
Export task started for Dhamtari.
Export task started for Durg.
Export task started for Gariaband.
Export task started for Janjgir_Champa.
Export task started for Jashpur.
Export task started for Kabeerdham.
Export task started for Kondagaon.
Export task started for Korba.
Export task started for Koriya.
Export task started for Mahasamund.
Export 

Check if every district image was exported

In [ ]:
gee_folder = 'projects/ee-mtpictd/assets/anoushka/LTP_area_perim_2023'

assets = ee.data.listAssets({'parent': gee_folder})['assets']

# Get only the base names of the assets (district name)
asset_names = set([asset['name'].split('/')[-1] for asset in assets])

count = 0

for acz in agroClimaticZones:
    print(acz)
    distDf = pd.read_csv(f'drive/MyDrive/Anoushka/ACZ_Districts/{acz}.csv')
    distList = list(distDf['Name'])
    for i, district in enumerate(distList):
      district_names.add(district)
      count += 1
      clean_name = district.replace('&', 'and').replace('.', '')
      clean_name = ''.join(c if c.isalnum() else '_' for c in clean_name)

      if clean_name not in asset_names:
          print(f"{district} not found")

print('Total number of districts:', count)

Eastern Plateau & Hills Region
Southern Plateau and Hills Region
East Coast Plains & Hills Region
Western Plateau and Hills Region
Central Plateau & Hills Region
Lower Gangetic Plain Region
Middle Gangetic Plain Region
Eastern Himalayan Region
Western Himalayan Region
Upper Gangetic Plain Region
Trans Gangetic Plain Region
West Coast Plains & Ghat Region
Gujarat Plains & Hills Region
Western Dry Region
Total number of districts: 897


# Given LTP mask, export distance-from-forest-edge raster state-wise

Unlike LTP mask, Distance from Forest Edge is a fast enough computation that it can be done at the state level instead of the district level. In the code below, I took India's state boundaries from GAUL, which contains the international map of India (excludes J&K, doesn't have Telangana.)

As a result I had to compute an additional 'diff' (difference) image for the areas not included in GAUL's map of India. But that can be avoided if you use a more accurate map of India's state boundaries! Just edit the list of `states` below and change the function `get_state_roi` in Utils.

In [ ]:
# State names
# J&K is missing; GAUL doesn't have a state boundary for it
states = [
    'andhra_pradesh', 'arunachal_pradesh', 'assam', 'bihar', 'chhattisgarh',
    'gujarat', 'haryana', 'himachal_pradesh', 'jharkhand',
    'delhi', 'chandigarh', 'goa',
    'karnataka', 'kerala', 'madhya_pradesh', 'maharashtra', 'manipur',
    'meghalaya', 'mizoram', 'nagaland', 'orissa', 'punjab', 'rajasthan',
    'sikkim', 'tamil_nadu', 'telangana', 'tripura', 'uttar_pradesh',
    'uttarakhand', 'west_bengal'
]

# Mapping state names to GAUL's nomenclature
gaulNames = {
    'andhra_pradesh': 'Andhra Pradesh',
    'arunachal_pradesh': 'Arunachal Pradesh',
    'assam': 'Assam',
    'bihar': 'Bihar',
    'chhattisgarh': 'Chhattisgarh',
    'gujarat': 'Gujarat',
    'haryana': 'Haryana',
    'himachal_pradesh': 'Himachal Pradesh',
    'jharkhand': 'Jharkhand',
    'delhi': 'Delhi',
    'chandigarh': 'Chandigarh',
    'goa': 'Goa',
    'karnataka': 'Karnataka',
    'kerala': 'Kerala',
    'madhya_pradesh': 'Madhya Pradesh',
    'maharashtra': 'Maharashtra',
    'manipur': 'Manipur',
    'meghalaya': 'Meghalaya',
    'mizoram': 'Mizoram',
    'nagaland': 'Nagaland',
    'orissa': 'Orissa',
    'punjab': 'Punjab',
    'rajasthan': 'Rajasthan',
    'sikkim': 'Sikkim',
    'tamil_nadu': 'Tamil Nadu',
    'telangana': 'Andhra Pradesh', # GAUL 2015 is not updated to reflect Telangana
    'tripura': 'Tripura',
    'uttar_pradesh': 'Uttar Pradesh',
    'uttarakhand': 'Uttarakhand',
    'west_bengal': 'West Bengal'
}

## Import LTP mask

In [ ]:
ltp_districts_2023 = get_img_collection('projects/ee-mtpictd/assets/anoushka/LTP_area_perim_2023')
ltp_india_2023 = ltp_districts_2023.mosaic().clip(india.geometry())

NameError: name 'get_img_collection' is not defined

## Computate distance to forest edge for both forest and non-forest pixels

In [ ]:
# Distance of non-forested pixels to forest edge
dist_img_nonforest = ltp_india_2023.select('large_tree_patch').fastDistanceTransform() \
        .sqrt() \
        .multiply(ee.Image.pixelArea().sqrt()) \
        .reproject(crs='EPSG:4326', scale=scale) \
        .selfMask()

# Invert LTP to get 'unforested' mask
invert_ltp = ee.Image(1).where(ltp_india_2023.select('large_tree_patch').eq(1), 0).clip(india.geometry()).selfMask()

# Distance of forested pixels to forest edge
dist_img_forest = invert_ltp.fastDistanceTransform() \
        .sqrt() \
        .multiply(ee.Image.pixelArea().sqrt()) \
        .reproject(crs='EPSG:4326', scale=scale) \
        .selfMask()

## Loop to export a 2-band image for each state of India

In [ ]:
for state in states:
  roi = get_state_roi(state)
  dist_img_forest_state = dist_img_forest.clip(roi.geometry()).rename('dist_forest_edge_forest')
  dist_img_nonforest_state = dist_img_nonforest.clip(roi.geometry()).rename('dist_forest_edge_nonforest')
  combined = dist_img_forest_state.addBands(dist_img_nonforest_state).reproject(crs='EPSG:4326', scale=scale)
  combined.set('state', state)
  combined.set('year', year)
  task = ee.batch.Export.image.toAsset(
    image=combined,
    description=f'Dist_forest_edge_{state}',
    assetId=f'projects/ee-mtpictd/assets/anoushka/forest_edge_dist_2023/{state}',
  )
  task.start()
  print(f'Export started for {state}')


## Visualization

In [ ]:
visparams = {
    'min': 0,
    'max': 5000,  # meters (adjust based on the expected maximum distance in your image)
    'palette': ['006400', '7FFF00', 'FFFF00', 'FFA500', 'FF0000']  # green to red
}

In [ ]:
Map = geemap.Map()
Map.addLayer(ltp_india_2023.select('large_tree_patch'), {'palette': ['green']}, 'LTP India')
# Map.addLayer(ltp_india_2023.select('area_m2'), {'min': 9500, 'max': 146000000, 'palette': ['teal', 'blue']}, 'Area Kolar')
# Map.addLayer(ltp_india_2023.select('perimeter_m'), {'min': 380, 'max': 1050000,'palette': ['orange', 'red']}, 'Perimeter Kolar')

Map.centerObject(india, 4)
Map

## Later computation for 'diff' area not included in GAUL

In [ ]:
state_roi_list = []
for state in states:
  roi = get_state_roi(state)
  state_roi_list.append(roi)

state_rois = ee.Feature(ee.FeatureCollection(state_roi_list).union().first())
india_roi = ee.Feature(indiaACZs.filter(ee.Filter.inList('regionname', ['Eastern Himalayan Region', 'Western Himalayan Region', 'Trans Gangetic Plain Region', 'Gujarat Plains & Hills Region'])).union().first())
diff = india_roi.difference(state_rois)

NameError: name 'get_state_roi' is not defined

In [ ]:
state = 'diff'
roi = diff
dist_img_forest_state = dist_img_forest.clip(roi.geometry()).rename('dist_forest_edge_forest')
dist_img_nonforest_state = dist_img_nonforest.clip(roi.geometry()).rename('dist_forest_edge_nonforest')
combined = dist_img_forest_state.addBands(dist_img_nonforest_state).reproject(crs='EPSG:4326', scale=scale)
combined.set('state', state)
combined.set('year', year)


In [ ]:
Map = geemap.Map()

visparams = {
    'min': 0,
    'max': 15000,  # meters (adjust based on the expected maximum distance in your image)
    'palette': ['006400', '7FFF00', 'FFFF00', 'FFA500', 'FF0000']  # green to red
}
# Map.addLayer(ltp_img.select('large_tree_patch'), {'palette': ['green']}, 'LTP')
Map.addLayer(combined.select('dist_forest_edge_forest'), visparams, 'Forested')
Map.addLayer(combined.select('dist_forest_edge_nonforest'), visparams, 'Non-forested')
# Map.addLayer(diff.geometry(), {}, 'Geom')
Map.centerObject(india, 4)
Map

In [ ]:
task = ee.batch.Export.image.toAsset(
  image=combined,
  description=f'Dist_forest_edge_{state}',
  assetId=f'projects/ee-mtpictd/assets/anoushka/forest_edge_dist_2023/{state}',
  maxPixels=1e13
)
task.start()
print(f'Export started for {state}')


# Given LTP mask, export interpatch distance of LTP polygons district-wise

This code generates a raster defined only on LTP pixels, where the value of a pixel in a Large Tree Patch (LTP) is the distance of that patch to its nearest neighbouring LTP. The main work is done in the function `interpatch_dist_image(feature, roi_buffer, ltp_buffer)`.


Due to computational constraints, this calculation has to be done at a district level. So LTPs near the district boundary might have overestimated values (if there is a closer LTP in the next district). To mitigate this, we search in an ROI of the district boundary + `roi_buffer`. I used `roi_buffer` = 1000 m.


Also due to computational constraints, it is not possible to compare each LTP in a district against every other LTP. So we search for the nearest neighbour within `ltp_buffer` meters. I used `ltp_buffer` = 1000 m.



In [ ]:
collection = get_img_collection('projects/ee-mtpictd/assets/anoushka/LTP_area_perim_2023')
ltp_mask_india = collection.select('large_tree_patch').mosaic()

In [ ]:
# Helper function to index a vector of polygons
def set_polygons_index(polygons):
  polygons_list = polygons.toList(polygons.size())
  index_list = ee.List.sequence(0, polygons.size().subtract(1))
  zipped = index_list.zip(polygons_list)

  def set_index(index_feature):
      index_feature = ee.List(index_feature)
      index = index_feature.get(0)
      feature = ee.Feature(index_feature.get(1))
      return feature.set('index', index)

  return ee.FeatureCollection(zipped.map(set_index))

# Given a feature (a district), get all LTP polygons in that feature + buffer
# Calculate nearest patch distance for each polygon and rasterize result
def interpatch_dist_image(feature, roi_buffer, ltp_buffer):
  roi = feature.buffer(roi_buffer)
  ltp_mask = ltp_mask_india.clip(roi.geometry())
  ltp_polygons = ltp_mask.reduceToVectors(
      crs='EPSG:4326',
      scale=scale,
      geometry=roi.geometry(),
      maxPixels=1e13,
      geometryType='polygon',
      bestEffort=True
      )
  ltp_polygons_with_index = set_polygons_index(ltp_polygons)

  def set_interpatch_dist(polygon):
      this_index = polygon.get('index')
      geom = polygon.geometry()
      buffered_geom = geom.buffer(ltp_buffer)
      others = ltp_polygons_with_index \
          .filter(ee.Filter.neq('index', this_index)) \
          .filterBounds(buffered_geom)

      def set_distance(other):
          dist = geom.distance(other.geometry(), scale)
          return other.set('dist', dist)
      distances = others.map(set_distance)

      count = distances.size()
      # If count is zero, set a default distance, else compute min
      min_dist = ee.Algorithms.If(
          count.eq(0),
          ltp_buffer + 50,
          ee.Feature(distances.sort('dist').first()).get('dist')
      )
      return polygon.set('interpatch_dist_m', min_dist)

  interpatch_dist = ltp_polygons_with_index.map(set_interpatch_dist)

  dist_img = interpatch_dist.reduceToImage(
    properties=['interpatch_dist_m'],
    reducer=ee.Reducer.first()
  ).rename('interpatch_dist_m')

  return dist_img.clip(feature.geometry()).reproject(crs='EPSG:4326', scale=scale)



In [ ]:
district_list = indiaDistricts.toList(indiaDistricts.size()).getInfo()
count = 0
for d in district_list:
  geom = ee.Feature(d)
  district_name = d["properties"]["Name"]  # adjust property name as needed

  district = district_name.replace('&', 'and').replace('.', '')
  district = re.sub(r'[^a-zA-Z0-9]', '_', district)

  geom = geom.set({'district' : district})
  interpatch_img = interpatch_dist_image(geom, 1000, 1000)
  task = ee.batch.Export.image.toAsset(
      image=interpatch_img,
      description=f'LTP_Interpatch_Dist_{district}',
      assetId=f'projects/ee-ictd-dhruvi/assets/anoushka/forest_interpatch_dist_2023/{district}',
  )
  task.start()
  print(f'Export task started for {district}.')
  count += 1
  time.sleep(0.25)

print(count)

Export task started for Nicobar_Islands.
Export task started for North_and_Middle_Andaman.
Export task started for South_Andaman.
Export task started for Anantapur.
Export task started for Chittoor.
Export task started for East_Godavari.
Export task started for Guntur.
Export task started for Krishna.
Export task started for Kurnool.
Export task started for Nellore.
Export task started for Prakasam.
Export task started for Srikakulam.
Export task started for Visakhapatnam.
Export task started for Vizianagaram.
Export task started for West_Godavari.
Export task started for YSR.
Export task started for Anjaw.
Export task started for Changlang.
Export task started for Dibang_Valley.
Export task started for East_Kameng.
Export task started for East_Siang.
Export task started for Kurung_Kumey.
Export task started for Lohit.
Export task started for Longding.
Export task started for Lower_Dibang_Valley.
Export task started for Lower_Subansiri.
Export task started for Namsai.
Export task start

# Corrected Interpatch Distance Images

In [7]:
interpatch_dist_img = get_img_collection("projects/ee-mtpictd/assets/anoushka/forest_interpatch_dist_2023").mosaic().clip(india.geometry())


In [13]:
def getACZDistricts(acz):
  distDf = pd.read_csv(f'drive/MyDrive/Anoushka/ACZ_Districts/{acz}.csv')
  distList = list(distDf['Name'])
  distRoiList = []
  for i, district in enumerate(distList):
    geom = ee.Feature(indiaDistricts.filter(ee.Filter.eq('Name', district)).first())
    distRoiList.append(geom)
  return distRoiList


In [ ]:
Map = geemap.Map()

visparams = {
    'min': 0,
    'max': 1050,  # meters (adjust based on the expected maximum distance in your image)
    'palette': ['006400', '7FFF00', 'FFFF00', 'FFA500', 'FF0000']  # green to red
}

Map.addLayer(interpatch_dist_img, visparams, 'Forest Interpatch Dist India')

# Map.addLayer(india, {}, 'India Boundary')
# Map.addLayer(aczDistricts, {}, f'{acz} Districts')
for acz in [
    'Eastern Himalayan Region',
    'Western Himalayan Region',
    'West Coast Plains & Ghat Region',
    'Southern Plateau and Hills Region'
    ]:
  districts = getACZDistricts(acz)
  Map.addLayer(districts, {}, f'{acz} Districts')
Map.centerObject(india, 4)
Map

FileNotFoundError: [Errno 2] No such file or directory: 'drive/MyDrive/Anoushka/ACZ_Districts/Eastern Himalayan Region.csv'

In [17]:
def correct_interpatch_dist(feature, threshold, ltp_buffer):
    # Clip inputs to district
    img = interpatch_dist_img.clip(feature.geometry())
    ltp_mask = ltp_mask_india.clip(feature.geometry())

    # Vectorize LTP polygons
    ltp_polygons = ltp_mask.selfMask().reduceToVectors(
        geometry=feature.geometry(),
        scale=scale,
        geometryType='polygon',
        maxPixels=1e13,
        bestEffort=True
    )

    # Correct polygons if needed
    def correct_polygon(poly):
        geom = poly.geometry()

        # Min value inside polygon
        min_inside = img.reduceRegion(
            reducer=ee.Reducer.min(),
            geometry=geom,
            scale=scale,
            maxPixels=1e13
        ).values().get(0)

        min_inside = ee.Number(min_inside)

        # Only correct if min > 1000
        def apply_correction():
            buffer_geom = geom.buffer(ltp_buffer)

            valid_buffer = img.updateMask(
                img.lte(threshold)   # <=threshold AND unmasked
            )

            buffer_min = valid_buffer.reduceRegion(
                reducer=ee.Reducer.min(),
                geometry=buffer_geom,
                scale=scale,
                maxPixels=1e13
            ).values().get(0)

            return poly.set('fill_val', buffer_min)

        return ee.Algorithms.If(
            min_inside.gt(threshold),
            apply_correction(),
            poly.set('fill_val', None)
        )

    corrected_polygons = ee.FeatureCollection(
        ltp_polygons.map(correct_polygon)
    ).filter(ee.Filter.notNull(['fill_val']))

    n_corrected = corrected_polygons.size()

    corrected_raster = ee.Image().paint(
      featureCollection=corrected_polygons,
      color='fill_val'
  )

    final_img = img.where(
        corrected_raster.mask(),
        corrected_raster
    )
    return final_img, n_corrected


In [14]:
collection = get_img_collection('projects/ee-mtpictd/assets/anoushka/LTP_area_perim_2023')
ltp_mask_india = collection.select('large_tree_patch').mosaic()

In [15]:
acz = 'Eastern Himalayan Region'
districts = getACZDistricts(acz)

In [18]:
for district in districts:
  district_name = district.get('Name').getInfo()
  district_name = district_name.replace('&', 'and').replace('.', '')
  district_name = re.sub(r'[^a-zA-Z0-9]', '_', district_name)

  roi = district.geometry()
  corrected_img, n_corrected = correct_interpatch_dist(district, 1000, 1000)
  if n_corrected.getInfo() > 0:
      task = ee.batch.Export.image.toAsset(
          image=corrected_img,
          description=f'Corrected_Interpatch_{district_name}',
          assetId=f'projects/ee-mtpictd/assets/anoushka/corrected_interpatch/{district_name}',
      )
      task.start()
      print(f'Export task started for {district_name}.')
  else:
    print(f'Skipping {district_name}: no corrections needed')


EEException: Computation timed out.

# Forest Cover Masks


Masks:
* Forested since 1985
* Forested since 2000
* Forested with disturbances
* Wholly deforested - under tree cover in 1985-2000, under tree cover at some time since 2000
* Fragmented - under tree cover in 1985-2000, under tree cover at some time since 2000

Datasets:
* Pre-2017 - GLC
* 2017-2022 - GLC and DW and IS
* post-2022 - DW and IS



## Get GLC tree cover

Helper function to return GLC tree cover in a given year - corresponds to get_dw_tree_cover and get_is_tree_cover in Utils

In [ ]:
def get_glc_tree_cover(year, roi, scale, annual=True):
  # LULC classes corresponding to tree cover
  forest_classes = [51, 52, 61, 62, 71, 72, 81, 82, 91, 92]
  # GLC is available annually from 2000 onwards
  if annual:
    band_num = year-1999
    glc_imgs = ee.ImageCollection("projects/sat-io/open-datasets/GLC-FCS30D/annual")
  else:
  # GLC is available every five years in 1985-2000
    band_num = (year-1980) // 5
    glc_imgs = ee.ImageCollection("projects/sat-io/open-datasets/GLC-FCS30D/five-years-map")
  band = 'b' + str(band_num)
  glc_img = glc_imgs.select(band).mosaic().clip(roi.geometry())
  mask = glc_img.remap(forest_classes, [1]*len(forest_classes)).eq(1)
  glc_img = glc_img.updateMask(mask)
  return glc_img.reproject(crs='EPSG:4326', scale=scale).rename('tree_cover')


## Forested area masks

This function returns a mask of all areas that have been under tree cover in every year in the array years. Tree cover is defined as:
* Pre-2017 - GLC
* 2017-2022 - GLC and DW and IS
* post-2022 - DW and IS

In [ ]:
def get_forested_area_intersection(years, roi, scale, verbose=False, glc=True, dw=True, isat=True):
  tree_cover_list = []
  # Collect all tree cover maps for each year for each applicable dataset
  for year in years:
    # In 1985-2000, use GLC Five Year
    if year in range(1985, 2000):
      if glc:
        tree_cover_list.append(get_glc_tree_cover(year, roi, scale, annual=False).set('dataset', 'GLC').set('year', year))
      else:
        print(f'No datasets for {year}')

    # In 2000-2015, use GLC annual
    elif year in range(2000, 2016):
      if glc:
        tree_cover_list.append(get_glc_tree_cover(year, roi, scale, annual=True).set('dataset', 'GLC').set('year', year))
      else:
        print(f'No datasets for {year}')

    # In 2016-2018, use GLC annual and Dynamic World
    elif year in range(2016, 2019):
      if glc:
        tree_cover_list.append(get_glc_tree_cover(year, roi, scale, annual=True).set('dataset', 'GLC').set('year', year))
      if dw:
        tree_cover_list.append(get_dw_tree_cover(roi, ee.Date(str(year)+'-01-01'), ee.Date(str(year+1)+'-01-01'), scale) \
                             .set('dataset', 'DW').set('year', year))
      if not glc and not dw:
        print(f'No datasets for {year}')

    # In 2019-2022, use GLC annual, DW, and in-lab IndiaSAT LULC
    elif year in range(2019, 2023):
      if glc:
        tree_cover_list.append(get_glc_tree_cover(year, roi, scale, annual=True).set('dataset', 'GLC').set('year', year))
      if dw:
        tree_cover_list.append(get_dw_tree_cover(roi, ee.Date(str(year)+'-01-01'), ee.Date(str(year+1)+'-01-01'), scale) \
                             .set('dataset', 'DW').set('year', year))
      if isat:
        tree_cover_list.append(get_is_tree_cover(roi, year-1, scale).set('dataset', 'IS').set('year', year))
      if not glc and not dw and not isat:
        print(f'No datasets for {year}')

    # 2023 onwards, use DW and IndiaSAT
    elif year >= 2023:
      if dw:
        if year == max(years):
          tree_cover_list.append(get_dw_tree_cover(roi, ee.Date(str(year)+'-01-01'), ee.Date(str(year)+'-07-01'), scale) \
                             .set('dataset', 'DW').set('year', year))
        else:
          tree_cover_list.append(get_dw_tree_cover(roi, ee.Date(str(year)+'-01-01'), ee.Date(str(year+1)+'-01-01'), scale) \
                             .set('dataset', 'DW').set('year', year))
      if isat:
        tree_cover_list.append(get_is_tree_cover(roi, year-1, scale).set('dataset', 'IS').set('year', year))
      if not isat and not dw:
        print(f'No datasets for {year}')

    else:
      print(f'No datasets for {year}')

  # Intersect with present day 'Forest Cover' or LTP mask
  # This gives current day forests that have been under tree cover in each year in years
  ltp_img = get_img_collection('projects/ee-mtpictd/assets/anoushka/LTP_area_perim_2023') \
            .select('large_tree_patch').mosaic().clip(roi.geometry())

  tree_cover_list.append(ltp_img.rename('current_tree_cover').set('year', 1000))

  # Intersect all tree cover images with current day LTP mask
  for i, img in zip([i for i in range(len(tree_cover_list))], tree_cover_list):
    if verbose:
      print(i, end = ' ')
      print(img.get('year').getInfo(), img.get('dataset').getInfo(), end=' ')
      print(img.bandNames().getInfo())
    ltp_img = ltp_img.And(img)
    # print(ltp_img.getInfo())

  # year = 0000 (special value) for the final tree cover image
  tree_cover_list.append(ltp_img.rename('final_tree_cover').set('year', 0000))

  return tree_cover_list

## Export forested area masks by acz

In [ ]:
for acz in agroClimaticZones:
  roi = indiaACZs.filter(ee.Filter.eq('regionname', acz))
  acronym = aczAcronymDict[acz]
  since_1985 = [1985, 1990, 1995] + [i for i in range(2000, 2024)]
  since_2000 = [i for i in range(2000, 2024)]
  betw_1985_2000 = [1985, 1990, 1995, 2000]

  # The last image returned in this list is the final_tree_cover
  img1 = get_forested_area_intersection(since_1985, roi, scale)[-1]

  task1 = ee.batch.Export.image.toAsset(
      image=img1,
      description=f'Forested_since_1985_{acronym}',
      assetId=f'projects/ee-ictd-dhruvi/assets/anoushka/forested_since_1985/{acronym}',
      region=roi.geometry(),
      scale=scale,
      crs='EPSG:4326',
      maxPixels=1e13
  )
  task1.start()
  print(f'Export task 1 started for {acronym}.')

  # The last image returned in this list is the final_tree_cover
  img2 = get_forested_area_intersection(since_2000, roi, scale)[-1]

  task2 = ee.batch.Export.image.toAsset(
      image=img2,
      description=f'Forested_since_2000_{acronym}',
      assetId=f'projects/ee-ictd-dhruvi/assets/anoushka/forested_since_2000/{acronym}',
      region=roi.geometry(),
      scale=scale,
      crs='EPSG:4326',
      maxPixels=1e13
  )
  task2.start()
  print(f'Export task 2 started for {acronym}.')


Export task 1 started for EPAHR.
Export task 2 started for EPAHR.
Export task 1 started for SPAHR.
Export task 2 started for SPAHR.
Export task 1 started for ECPHR.
Export task 2 started for ECPHR.
Export task 1 started for WPAHR.
Export task 2 started for WPAHR.
Export task 1 started for CPAHR.
Export task 2 started for CPAHR.
Export task 1 started for LGPR.
Export task 2 started for LGPR.
Export task 1 started for MGPR.
Export task 2 started for MGPR.
Export task 1 started for EHR.
Export task 2 started for EHR.
Export task 1 started for WHR.
Export task 2 started for WHR.
Export task 1 started for UGPR.
Export task 2 started for UGPR.
Export task 1 started for TGPR.
Export task 2 started for TGPR.
Export task 1 started for WCPGR.
Export task 2 started for WCPGR.
Export task 1 started for GPHR.
Export task 2 started for GPHR.
Export task 1 started for WDR.
Export task 2 started for WDR.


## Deforested areas mask

This function returns a mask of all areas that were under tree cover in any year in the array years. Tree cover is defined as:
* Pre-2017 - GLC
* 2017-2022 - GLC and DW and IS
* post-2022 - DW and IS

In [ ]:
# Returns areas that were forested at any time in the span of 'years'
# Setting verbose will slow things down considerably

def get_forested_area_union(years, roi, scale, verbose=False, glc=True, dw=True, isat=True):
  # Collect all tree cover masks for every year in years
  tree_cover_list = []
  for year in years:

    if year in range(1985, 2000):
      if glc:
        tree_cover_list.append(get_glc_tree_cover(year, roi, scale, annual=False).set('dataset', 'GLC').set('year', year))
      else:
        print(f'No datasets for {year}')

    elif year in range(2000, 2016):
      if glc:
        tree_cover_list.append(get_glc_tree_cover(year, roi, scale, annual=True).set('dataset', 'GLC').set('year', year))
      else:
        print(f'No datasets for {year}')

    elif year in range(2016, 2019):
      if glc:
        tree_cover_list.append(get_glc_tree_cover(year, roi, scale, annual=True).set('dataset', 'GLC').set('year', year))
      if dw:
        tree_cover_list.append(get_dw_tree_cover(roi, ee.Date(str(year)+'-01-01'), ee.Date(str(year+1)+'-01-01'), scale) \
                             .set('dataset', 'DW').set('year', year))
      if not glc and not dw:
        print(f'No datasets for {year}')

    elif year in range(2019, 2023):
      if glc:
        tree_cover_list.append(get_glc_tree_cover(year, roi, scale, annual=True).set('dataset', 'GLC').set('year', year))
      if dw:
        tree_cover_list.append(get_dw_tree_cover(roi, ee.Date(str(year)+'-01-01'), ee.Date(str(year+1)+'-01-01'), scale) \
                             .set('dataset', 'DW').set('year', year))
      if isat:
        tree_cover_list.append(get_is_tree_cover(roi, year-1, scale).set('dataset', 'IS').set('year', year))
      if not glc and not dw and not isat:
        print(f'No datasets for {year}')

    elif year >= 2023:
      if dw:
        if year == max(years):
          tree_cover_list.append(get_dw_tree_cover(roi, ee.Date(str(year)+'-01-01'), ee.Date(str(year)+'-07-01'), scale) \
                             .set('dataset', 'DW').set('year', year))
        else:
          tree_cover_list.append(get_dw_tree_cover(roi, ee.Date(str(year)+'-01-01'), ee.Date(str(year+1)+'-01-01'), scale) \
                             .set('dataset', 'DW').set('year', year))
      if isat:
        tree_cover_list.append(get_is_tree_cover(roi, year-1, scale).set('dataset', 'IS').set('year', year))
      if not isat and not dw:
        print(f'No datasets for {year}')

    else:
      print(f'No datasets for {year}')

  tree_cover_img = tree_cover_list[0]
  # Take union over all tree cover masks
  # To get pixels that have been under tree cover at any point in the span of years
  for i, img in zip([i for i in range(len(tree_cover_list))], tree_cover_list):
    if verbose:
      print(i, end = ' ')
      print(img.get('year').getInfo(), img.get('dataset').getInfo(), end=' ')
      print(img.bandNames().getInfo())
    tree_cover_img = tree_cover_img.Or(img)

  tree_cover_list.append(tree_cover_img.rename('final_tree_cover').set('year', 0000))

  return tree_cover_list


## Export acz wise

In [ ]:
for acz in agroClimaticZones:
  roi = indiaACZs.filter(ee.Filter.eq('regionname', acz))
  acronym = aczAcronymDict[acz]
  years_1985_2000 = [1985, 1990, 1995, 2000]
  years_2000_onwards = [i for i in range(2000,2024)]

  img1 = get_forested_area_union(years_1985_2000, roi, scale)[-1]

  task1 = ee.batch.Export.image.toAsset(
      image=img1,
      description=f'Forested_1985_2000_union_{acronym}',
      assetId=f'projects/ee-ictd-dhruvi/assets/anoushka/forested_1985_2000_union/{acronym}',
      region=roi.geometry(),
      scale=scale,
      crs='EPSG:4326',
      maxPixels=1e13
  )
  task1.start()
  print(f'Export task 1 started for {acronym}.')

  img2 = get_forested_area_union(years_2000_onwards, roi, scale)[-1]

  task2 = ee.batch.Export.image.toAsset(
      image=img2,
      description=f'Forested_2000_onwards_union_{acronym}',
      assetId=f'projects/ee-ictd-dhruvi/assets/anoushka/forested_2000_onwards_union/{acronym}',
      region=roi.geometry(),
      scale=scale,
      crs='EPSG:4326',
      maxPixels=1e13
  )
  task2.start()
  print(f'Export task 2 started for {acronym}.')

Export task 1 started for EPAHR.
Export task 2 started for EPAHR.
Export task 1 started for SPAHR.
Export task 2 started for SPAHR.
Export task 1 started for ECPHR.
Export task 2 started for ECPHR.
Export task 1 started for WPAHR.
Export task 2 started for WPAHR.
Export task 1 started for CPAHR.
Export task 2 started for CPAHR.
Export task 1 started for LGPR.
Export task 2 started for LGPR.
Export task 1 started for MGPR.
Export task 2 started for MGPR.
Export task 1 started for EHR.
Export task 2 started for EHR.
Export task 1 started for WHR.
Export task 2 started for WHR.
Export task 1 started for UGPR.
Export task 2 started for UGPR.
Export task 1 started for TGPR.
Export task 2 started for TGPR.
Export task 1 started for WCPGR.
Export task 2 started for WCPGR.
Export task 1 started for GPHR.
Export task 2 started for GPHR.
Export task 1 started for WDR.
Export task 2 started for WDR.
